# Notebook 05 — Business Insights

## Notebook Objective

This notebook serves as the final analytical stage of the project, bridging technical findings and business decision-making.

Rather than performing additional exploratory analysis, the focus is on answering key business questions by integrating results from Exploratory Data Analysis (EDA) and Customer Segmentation using RFM analysis.

The objective is to identify strategic opportunities, highlight customer and product performance, and provide data-driven recommendations that support business decision-making.

---

### Notebook Roadmap

This notebook addresses the following business questions:

- [x] Which customers generate the highest business value?
- [x] Which products contribute the most to revenue?
- [x] Which countries drive the largest share of sales?
- [x] Which customer segments should be prioritized?
- [x] What strategic actions can improve business performance?
- [x] What are the key conclusions from the overall analysis?

---

### Expected Outcome

By the end of this notebook, the analysis will be translated into clear business insights and practical recommendations that can support marketing, customer retention, and revenue growth strategies.

## 1. Import Libraries

The required libraries for data manipulation and visualization are imported below. These libraries will be used throughout the notebook to support the analysis and communicate business insights effectively.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)

# Plot settings
plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# 2. Load Processed Datasets

This notebook combines two processed datasets created in previous stages of the project.

- The cleaned transaction dataset provides detailed information about customer purchases.
- The customer-level RFM dataset contains segmentation results and customer value metrics.

These datasets are merged to create a unified analytical dataset for business insight generation.

In [ ]:
# Load the cleaned transaction dataset
transactions = pd.read_csv("../data/processed/online_retail_processed.csv")

# Load the customer RFM dataset
rfm = pd.read_csv("../data/processed/customer_rfm.csv")

In [ ]:
# Check the cleaned dataset information
transactions.info()

In [ ]:
# Check the RFM information
rfm.info()

## 3. Standardize Column Names

To ensure consistency across datasets and simplify the merging process, column names in the transactions dataset are standardized.

This step aligns naming conventions between the transaction-level dataset and the RFM dataset, preventing potential issues during data integration and improving code readability in subsequent analysis steps.

In [ ]:
transactions = transactions.rename(columns={
    'Invoice': 'InvoiceNo',
    'Customer ID': 'CustomerID',
    'Price': 'UnitPrice'
})

## 4. Merge Datasets

To support business-oriented analysis, the transaction-level dataset is combined with the customer-level RFM dataset.

The datasets are merged using the `CustomerID` field, allowing each transaction to be associated with its corresponding customer segment and RFM metrics. This unified dataset serves as the foundation for all subsequent business insights.

In [ ]:
# Merge transaction and RFM datasets
df = transactions.merge(
    rfm,
    on='CustomerID',
    how='left'
)

# Display the first five rows
df.head()

In [ ]:
df.shape

In [ ]:
df[['Recency', 'Frequency', 'Monetary']].isna().sum()

# 3. Analysis Framework

The insights presented in this notebook are derived from the analyses conducted in the previous notebooks, including Exploratory Data Analysis (EDA) and RFM-based customer segmentation.

Rather than introducing new analytical techniques, this notebook synthesizes existing findings to answer key business questions and translate analytical results into actionable business recommendations.

Each section follows a question-driven approach, where analytical evidence is used to support business decisions and strategic recommendations.

## 3.1 Customer Value Analysis

### Business Question

**Which customers generate the highest business value?**

### Objective

The objective of this section is to identify the customers who contribute the greatest value to the business. By combining purchasing behavior with RFM-based customer segmentation, we can better understand which customers drive long-term revenue and deserve the highest retention priority.

### 3.1.1 Customer Distribution by Segment

To understand where business value is concentrated, we first examine how customers are distributed across the RFM segments.

This provides context for interpreting revenue contribution and identifying whether high-value customer groups represent a large or small portion of the customer base.

In [ ]:
# Count customers in each RFM segment
segment_counts = (
    df['Segment']
    .value_counts()
    .sort_values()
)

# Create figure
fig, ax = plt.subplots(figsize=(8, 5))

# Plot
segment_counts.plot(
    kind='barh',
    ax=ax
)

# Labels
ax.set_title('Customer Distribution Across RFM Segments')
ax.set_xlabel('Number of Customers')
ax.set_ylabel('Customer Segment')

plt.tight_layout()
plt.show()

### 3.1.2 Revenue Contribution by Segment

To better understand the economic impact of each customer segment, we analyze how total revenue is distributed across RFM segments.

This helps identify which segments contribute the most to overall business performance, regardless of their size.

In [ ]:
# Calculate total revenue per segment
segment_revenue = df.groupby('Segment')['TotalPrice'].sum().sort_values()

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

segment_revenue.plot(kind='barh', ax=ax)

ax.set_title('Revenue Contribution by Customer Segment')
ax.set_xlabel('Total Revenue')
ax.set_ylabel('Customer Segment')

plt.tight_layout()
plt.show()

### Key Insight

This visualization shows how revenue is distributed across customer segments.

It highlights whether the business is dependent on a small group of high-value customers or whether revenue is more evenly distributed across segments.

Segments with higher revenue contribution represent the most economically important customer groups.

### 3.1.3 Average Customer Value by Segment

To better understand the true value of each customer segment, we analyze the average monetary value per customer.

This helps identify which segments are not only generating total revenue but are also the most valuable on an individual customer basis.

In [ ]:
# Calculate average monetary value per segment
segment_avg_value = df.groupby('Segment')['Monetary'].mean().sort_values()

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

segment_avg_value.plot(kind='barh', ax=ax)

ax.set_title('Average Customer Value by Segment')
ax.set_xlabel('Average Monetary Value')
ax.set_ylabel('Segment')

plt.tight_layout()
plt.show()

### Key Insight

This chart highlights the average value generated by each customer segment.

High-value segments such as Champions typically show significantly higher average monetary contribution, confirming their importance for revenue generation and long-term business growth.

Lower-value segments may represent potential opportunities for upselling, reactivation, or targeted marketing campaigns.

### 3.1 Conclusion: Customer Value Analysis

In this section, we analyzed customer value from three perspectives:
- Customer distribution across segments
- Total revenue contribution by segment
- Average customer value per segment

Together, these analyses provide a comprehensive view of how value is distributed across the customer base.

### Key Insights

- A small number of high-value segments contribute a disproportionate share of total revenue.
- Customer distribution is not necessarily aligned with revenue contribution.
- High-value segments demonstrate significantly higher average monetary value per customer.

Overall, the business relies heavily on a concentrated group of valuable customers.

### Business Implications

- Retention strategies should prioritize high-value segments (e.g., Champions).
- Medium-value segments represent opportunities for growth through upselling and engagement strategies.
- Low-value or inactive segments may require reactivation campaigns or cost-efficient marketing approaches.

## 3.2 Product Performance Analysis

In this section, we analyze product-level performance to identify which products contribute the most to revenue.

This helps us understand which items drive business growth and should be prioritized in inventory management and marketing strategies.

### 3.2.1 Top Revenue-Generating Products

We first identify the products that generate the highest total revenue.

In [ ]:
# Calculate revenue per product
top_products = (
    df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values()
    .tail(10)
)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

top_products.plot(kind='barh', ax=ax)

ax.set_title('Top 10 Revenue-Generating Products')
ax.set_xlabel('Total Revenue')
ax.set_ylabel('Product')

plt.tight_layout()
plt.show()

### Key Insight

A small number of products generate a large share of total revenue.

These products are critical for business performance and should be prioritized in stock management, promotion strategies, and supply chain planning.

### 3.2.2 Most Frequently Purchased Products

In this section, we analyze product frequency to identify which items are purchased most often, regardless of their total revenue contribution.

This helps us understand customer demand patterns and identify popular products.

In [ ]:
# Calculate product frequency (number of times purchased)
top_freq_products = (
    df.groupby('Description')['Quantity']
    .sum()
    .sort_values()
    .tail(10)
)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

top_freq_products.plot(kind='barh', ax=ax)

ax.set_title('Top 10 Most Frequently Purchased Products')
ax.set_xlabel('Total Quantity Sold')
ax.set_ylabel('Product')

plt.tight_layout()
plt.show()

### Key Insight

This chart highlights the most frequently purchased products, reflecting customer demand patterns.

Some products may not generate the highest revenue individually, but their high purchase frequency indicates strong customer preference and consistent demand.

These products are important for inventory planning and maintaining customer satisfaction.

### 3.2.3 Product Performance Summary

In this section, we combine insights from both revenue and frequency analyses to better understand overall product performance.

This allows us to distinguish between high-revenue products and high-demand products, which may not always overlap.

In [ ]:
# Create product-level summary table
product_stats = df.groupby('Description').agg({
    'TotalPrice': 'sum',
    'Quantity': 'sum'
}).reset_index()

# Top 10 by revenue
top_revenue = product_stats.sort_values('TotalPrice', ascending=False).head(10)

# Top 10 by frequency
top_frequency = product_stats.sort_values('Quantity', ascending=False).head(10)

# Strategic products (intersection)
strategic_products = set(top_revenue['Description']) & set(top_frequency['Description'])

# Build final table
strategic_df = product_stats[product_stats['Description'].isin(strategic_products)]
strategic_df = strategic_df.sort_values('TotalPrice', ascending=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(strategic_df['Description'], strategic_df['TotalPrice'])

ax.set_title('Strategic Products (High Revenue + High Frequency)')
ax.set_xlabel('Total Revenue')
ax.set_ylabel('Product')

plt.tight_layout()
plt.show()

### Strategic Products Analysis — Key Insights

By combining revenue and purchase frequency perspectives, we identified a small group of products that are both highly purchased and high revenue-generating.

These products represent the core drivers of business performance.

### Key Insights

- Only a limited number of products appear in both top revenue and top frequency groups.
- These products indicate a strong overlap between customer demand and revenue generation.
- The business is partially dependent on a small set of high-performing products.

### Business Implications

- These strategic products should be prioritized in inventory management to avoid stockouts.
- Marketing efforts should focus on increasing visibility of these products to maximize revenue.
- Supply chain stability for these items is critical, as they directly impact overall business performance.
- Monitoring these products regularly is essential because any decline in their performance can significantly affect revenue.

## 3.3 Geographic Sales Insights

In this section, we analyze sales performance across different countries to understand the geographic distribution of revenue.

This helps identify key markets, potential dependencies, and opportunities for market expansion.

### 3.3.1 Revenue Contribution by Country

We first analyze total revenue generated by each country to identify the most important markets.

In [ ]:
# Calculate revenue by country
country_revenue = (
    df.groupby('Country')['TotalPrice']
    .sum()
    .sort_values()
)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

country_revenue.plot(kind='barh', ax=ax)

ax.set_title('Revenue Contribution by Country')
ax.set_xlabel('Total Revenue')
ax.set_ylabel('Country')

plt.tight_layout()
plt.show()

### Key Insight

Revenue is not evenly distributed across countries.

A small number of countries contribute the majority of total sales, indicating a concentrated market structure.

### 3.3.2 Customer Activity by Country

We now examine the number of transactions per country to understand customer activity distribution.

In [ ]:
# Count transactions by country
country_orders = df['Country'].value_counts().sort_values()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

country_orders.plot(kind='barh', ax=ax)

ax.set_title('Number of Transactions by Country')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Country')

plt.tight_layout()
plt.show()

### Key Insight

The number of transactions also varies significantly across countries.

This indicates that certain markets are not only generating higher revenue but also showing higher customer activity levels.

### 3.3 Conclusion: Geographic Sales Insights

In this section, we analyzed the geographic distribution of sales using two key perspectives:
- Total revenue by country
- Number of transactions by country

Together, these metrics help us understand both market value and customer activity across regions.

### Key Insights

- Sales are highly concentrated in a small number of countries.
- A few markets dominate both revenue generation and transaction volume.
- Some countries show high activity but relatively lower revenue contribution, indicating differences in purchasing power or order value.

### Business Implications

- The business is exposed to geographic concentration risk and should avoid over-dependence on a limited number of markets.
- High-revenue countries should be prioritized for retention and strategic partnerships.
- High-activity but lower-revenue markets represent potential growth opportunities through pricing, marketing, or product strategy adjustments.
- Market expansion strategies should focus on underrepresented but high-potential regions.

## 4. Executive Summary

This notebook aimed to transform analytical outputs from previous stages into actionable business insights.

By combining customer segmentation, product performance analysis, and geographic insights, we developed a comprehensive understanding of the business structure and key value drivers.

### Key Findings

- Customer value is highly concentrated within a small number of high-value segments.
- A limited set of products drives both the majority of revenue and customer demand.
- Sales are geographically concentrated in a few key markets.
- There are clear differences between customer volume, revenue contribution, and customer value across segments.

### Key Findings

- Customer value is highly concentrated within a small number of high-value segments.
- A limited set of products drives both the majority of revenue and customer demand.
- Sales are geographically concentrated in a few key markets.
- There are clear differences between customer volume, revenue contribution, and customer value across segments.

## 5. Strategic Actions for Business Improvement

Based on all analyses conducted in this project, several strategic actions can be recommended to improve overall business performance.

### Key Strategic Actions

Based on the analysis, the following strategic actions are recommended:

- Retention: Prioritize high-value segments (e.g., Champions) to maximize customer lifetime value.
- Product Strategy: Strengthen inventory management for high-revenue and high-demand products.
- Marketing: Develop targeted campaigns for mid-value customers to increase engagement.
- Geography: Reduce dependency on a few markets and explore expansion opportunities.
- Monitoring: Continuously track top-performing customers and products.

### Final Note

These actions directly translate analytical insights into business decisions and provide a roadmap for improving revenue, customer value, and market performance.

## 6. Save Final Customer Analysis Dataset

In [ ]:
df.to_csv("../data/processed/final_customer_analysis.csv", index=False)

print("Final customer analysis saved successfully.")